In [1]:
import numpy as np
import pandas as pd
import plotly.express as px

In [2]:
df_temperature = pd.read_csv("Source_Data/temperaturdaten.csv", encoding="utf-8", sep=";")
df_fitness =  pd.read_csv("Dataframes/workouts_per_second.csv", encoding="utf-8")
df_rides = pd.read_csv("Dataframes/ride_records.csv", encoding="utf-8")

In [3]:
df_rides.columns

Index(['source_file', 'message_type', 'manufacturer', 'number', 'product',
       'product_name', 'serial_number', 'time_created', 'type',
       'application_id', 'application_version', 'developer_data_index',
       'developer_id', 'manufacturer_id', 'field_definition_number',
       'field_name', 'fit_base_type_id', 'fit_base_unit_id',
       'native_field_num', 'native_mesg_num', 'offset', 'scale', 'units',
       'ant_device_number', 'ant_device_type', 'ant_network',
       'ant_transmission_type', 'battery_status', 'battery_voltage',
       'cum_operating_time', 'descriptor', 'device_index', 'hardware_version',
       'sensor_position', 'software_version', 'source_type', 'timestamp',
       'data16', 'event', 'event_group', 'event_type', 'front_gear',
       'front_gear_num', 'opponent_score', 'rear_gear', 'rear_gear_num',
       'score', 'timer_trigger', 'unknown_21', 'unknown_22', 'avg_cadence',
       'avg_heart_rate', 'avg_power', 'avg_speed', 'end_position_lat',
       'end_

In [7]:
df_rides.message_type.unique()

<ArrowStringArray>
[          'file_id', 'developer_data_id', 'field_description',
       'device_info',             'event',               'lap',
           'session',          'activity']
Length: 8, dtype: str

Der DataFrame df_temperature enthält Messdaten der Station München-Stadt für den Zeitraum vom 10. Februar 2025 bis zum 13. August 2025.

Jeder Datenpunkt umfasst die innerhalb der letzten zehn Minuten gemessene Höchst- und Tiefsttemperatur. Die Messungen wurden in einer Höhe von 2 Metern sowie 5 Zentimetern über dem Boden durchgeführt.

Für die weitere Analyse wird ausschließlich die Lufttemperatur in 2 Metern Höhe verwendet, da diese die Temperatur in meiner Wohnung vermutlich stärker beeinflusst. Die entsprechenden Werte sind in den Spalten TX_10 (Höchsttemperatur) und TN_10 (Tiefsttemperatur) enthalten.

In [ ]:
df_temperature["TM_10"] = (df_temperature["TX_10"] + df_temperature["TN_10"])/2
print(df_temperature)

In [ ]:
df_temperature.drop(columns=['TX5_10', 'TN5_10', 'eor'], inplace=True)
print(df_temperature)

Die Fitnessdateien enthalten für jede Sekunde eines Workouts einen Datenpunkt. Dabei werden zahlreiche Messwerte in unterschiedlichen Spalten gespeichert. Einige dieser Spalten werden von Zwift jedoch nicht befüllt und enthalten daher keine relevanten Daten.

Im Folgenden wird zunächst ein kurzer Überblick über die vorhandenen Spalten gegeben.


In [ ]:
print(df_fitness.describe())

In [ ]:
print(df_fitness.count() == 0)

In [ ]:
df_fitness.dropna(axis=1, how='all', inplace = True)
print(df_fitness.head())

In [ ]:
print(df_fitness[df_fitness['altitude'] != df_fitness['enhanced_altitude']])
print(df_fitness['speed_1s'].unique())

In [ ]:
df_fitness.drop(columns = ['enhanced_altitude', 'speed_1s'], inplace = True)
print(df_fitness.head())

In [ ]:
df_fitness.info()

In [ ]:
df_temperature.info()

In [ ]:
df_temperature.columns

In [ ]:
df_temperature.columns = df_temperature.columns.str.strip()
df_temperature.columns

In [ ]:
# 1. Datumsformate in echte datetime-Objekte umwandeln
df_temperature['MESS_DATUM'] = pd.to_datetime(df_temperature['MESS_DATUM'], format='%Y%m%d%H%M', utc=True)
df_fitness['timestamp'] = pd.to_datetime(df_fitness['timestamp'], format='%Y-%m-%d %H:%M:%S', utc=True)

# 2. Beide DataFrames nach der Zeit sortieren (Voraussetzung für merge_asof)
df_temperature = df_temperature.sort_values('MESS_DATUM')
df_fitness = df_fitness.sort_values('timestamp')

# 3. merge_asof: findet für jeden fitness-Zeitpunkt den passenden
#    MESS_DATUM-Wert, der <= timestamp ist (backward), 
#    aber maximal 10 Minuten zurückliegt (tolerance)
df_merged = pd.merge_asof(
    df_fitness,
    df_temperature[['MESS_DATUM', 'QN', 'TM_10']],
    left_on='timestamp',
    right_on='MESS_DATUM',
    direction='forward',
    tolerance=pd.Timedelta('10min')
)

In [ ]:
df_merged.sample(10)

In [ ]:
df_merged.describe()

In [ ]:
df_merged[df_merged.TM_10 == -999]

In [ ]:
df_merged["heart_rate"] = df_merged["heart_rate"].replace(0, np.nan)
df_merged["TM_10"] = df_merged["TM_10"].replace(-999, np.nan)
df_merged["target_power"] = df_merged["target_power"].replace(0, np.nan)

In [ ]:
df_merged.describe()

In [ ]:
df_merged["day_of_week"] = df_merged.timestamp.dt.day_name()
df_merged.sample(5)

Bei der Tageszeit muss beachtet werden, dass timestamp in UTC angegeben ist

In [ ]:
df_merged['timestamp_berlin'] = df_merged['timestamp'].dt.tz_convert('Europe/Berlin')

In [ ]:
import plotly.express as px

df_merged['hour_utc'] = df_merged['timestamp'].dt.hour

fig = px.histogram(df_merged, x='hour_utc', nbins=24, 
                    title='Verteilung nach Stunde')
fig.update_xaxes(dtick=1, title='Stunde des Tages')
fig.update_yaxes(title='Anzahl')
fig.show()

In [ ]:
df_merged['hour_berlin'] = df_merged['timestamp_berlin'].dt.hour

fig = px.histogram(df_merged, x='hour_berlin', nbins=24, 
                    title='Verteilung nach Stunde')
fig.update_xaxes(dtick=1, title='Stunde des Tages')
fig.update_yaxes(title='Anzahl')
fig.show()

In [ ]:
# Beispiel-Daten (ersetze 'timestamp' mit deiner Datums-/Zeit-Spalte)
df['time_of_day'] = df['timestamp'].dt.hour.apply(
    lambda hour:
        'Morning' if 6 <= hour < 12 else
        'Afternoon' if 12 <= hour < 18 else
        'Evening' if 18 <= hour < 22 else
        'Night'
)